# STAT 207 Homework 11 [25 points]

## Regularization Models for Linear Relationships

Due: Thursday, December 5, end of day (11:59 pm CT)

<hr>

## Imports 

Run the following code cell to import the necessary packages into the file.  You may import additional packages, as needed for this assignment.

In [247]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso

## The Data

With available climate data dating back many decades and the prevalence of climate change, humans are looking to understand exactly how different features of the climate affect the temperatures globally.  For this assignment, we will look to understand how the **global temperature** fluctuates based on other environmental features.

We will use various atmospheric and temperature measures over 309 months from 1983 to 2008, with the following variables:

- **Year**: the observation year
- **Month**: the observation month, recorded with numbers 1 to 12
- **MEI**: Multivariate El Nino Southern Oscillation Index (MEI), measuring the affects of the El Nino weather pattern
- **CO2**: atmospheric concentration of carbon dioxide (in ppmv, parts per million by volume)
- **CH4**: atmospheric concentration of methane (in ppmv)
- **N2O**: atmospheric concentration of nitrous oxide (in ppmv)
- **CFC-11**: atmospheric concentration of CCl3F or trichlorofluoromethane (in ppbv, parts per billion by volume)
- **CFC-12**: atmospheric concentration of CCl2F2 or dichlorodifluoromethane (in ppbv)
- **TSI**: the total solar irradiance (TSI) (in W/m2), measuring the rate at which the sun's energy is deposited per unit area.
- **Aerosols**: the mean stratospheric aerosol optical depth at 500 nm, a measure associated with volcanic activity
- **Temp**: the difference in the average global temperature for that month (in Celsius) and a reference value

The ESRL/NOAA Physical Sciences Division reports the MEI; atmospheric concentrations are measured by the ESRL/NOAA Global Monitoring Division; the SOLARIS-HEPPA project website provides the TSI; the Godard Institute for Space Studies at NASA reports the Aerosols; and the Climatic Research Unit at the University of East Anglia reports the Temp.

Run the code in the cell below to read in the cleaned data for this document.  The data is saved as `df` with this code.  

In [248]:
df = pd.read_csv('climate_change.csv')
df_train = df[df['Year'] <= 2006]
df_test = df[df['Year'] >= 2007]
X_train = df_train.drop(['Year', 'Month', 'Temp'], axis = 1)
X_test = df_test.drop(['Year', 'Month', 'Temp'], axis = 1)
y_train = df_train['Temp']
y_test = df_test['Temp']

## 1. Summarize Data [1.5 points]

Above, we set aside a training data.  We didn't randomly select our training and test set; instead, imagine that we fit a model using the available data in 2006 in our training data.  We'll then use the data that we collect in the following two years as the test set to evaluate this model.

As defined in our X_train and X_test above, our response variable for this assignment with be **Temp**.  We'll use all variables except the **Year** and **Month** as our predictor variables.

**a)** Scale our predictor variables in the training data.

In [249]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

**b)** Now, we want to be sure that we also scale our test data, using the same scaling as applied to our training data.  Apply your scaling algorithm from **part a** to the test data.  

*Note*: This does not include re-fitting your scaling process.  You will re-use your scaling process from **part a**, simply transforming your test data with the same scaling process.

In [250]:
X_test_scaled = scaler.transform(X_test)

In [251]:
X_train.mean().max()

1745.8414788732393

In [252]:
X_train.mean().min()

0.017720774647887325

In [253]:
X_train_scaled.mean().min()

-5.373948547506022e-14

In [254]:
means_train = pd.DataFrame(X_train_scaled, columns=X_train.columns).mean()
print(means_train)
stds_train = pd.DataFrame(X_train_scaled, columns=X_train.columns).std()
print(stds_train)
means_test = pd.DataFrame(X_test_scaled, columns=X_train.columns).mean()
print(means_test)
stds_test = pd.DataFrame(X_test_scaled, columns=X_train.columns).std()
print(stds_test)

MEI         3.752867e-17
CO2         2.501911e-15
CH4         2.301758e-15
N2O        -2.001529e-16
CFC-11     -1.150879e-15
CFC-12      2.501911e-16
TSI        -4.336813e-13
Aerosols    0.000000e+00
dtype: float64
MEI         1.001765
CO2         1.001765
CH4         1.001765
N2O         1.001765
CFC-11      1.001765
CFC-12      1.001765
TSI         1.001765
Aerosols    1.001765
dtype: float64
MEI        -0.917795
CO2         2.036889
CH4         1.121218
N2O         1.984680
CFC-11     -0.314865
CFC-12      0.720071
TSI        -0.982834
Aerosols   -0.455588
dtype: float64
MEI         0.690182
CO2         0.179041
CH4         0.250031
N2O         0.114226
CFC-11      0.065486
CFC-12      0.024218
TSI         0.079206
Aerosols    0.020206
dtype: float64


In [255]:
print(X_train.describe())
print(X_test.describe())

              MEI         CO2          CH4         N2O      CFC-11  \
count  284.000000  284.000000   284.000000  284.000000  284.000000   
mean     0.341923  361.414261  1745.841479  311.657225  252.487092   
std      0.929639   11.439691    45.669846    4.758513   20.987671   
min     -1.586000  340.170000  1629.890000  303.677000  191.324000   
25%     -0.323000  352.315000  1716.347500  307.657000  249.557750   
50%      0.308500  359.890000  1758.605000  310.849500  260.373500   
75%      0.898000  370.585000  1781.637500  316.129250  267.448000   
max      3.001000  384.980000  1808.150000  320.451000  271.494000   

           CFC-12          TSI    Aerosols  
count  284.000000   284.000000  284.000000  
mean   494.217546  1366.101437    0.017721  
std     59.046642     0.401283    0.030014  
min    350.113000  1365.426100    0.001600  
25%    462.543000  1365.754550    0.002700  
50%    522.089000  1366.054500    0.006200  
75%    540.972750  1366.399275    0.014000  
max    54

In [256]:
X_train.corr()

,MEI,CO2,CH4,N2O,CFC-11,CFC-12,TSI,Aerosols
MEI,1.000000,-0.041147,-0.033419,-0.050820,0.069000,0.008286,-0.154492,0.340238
CO2,-0.041147,1.000000,0.877280,0.976720,0.514060,0.852690,0.177429,-0.356155
CH4,-0.033419,0.877280,1.000000,0.899839,0.779904,0.963616,0.245528,-0.267809
N2O,-0.050820,0.976720,0.899839,1.000000,0.522477,0.867931,0.199757,-0.337055
CFC-11,0.069000,0.514060,0.779904,0.522477,1.000000,0.868985,0.272046,-0.043921
CFC-12,0.008286,0.852690,0.963616,0.867931,0.868985,1.000000,0.255303,-0.225131
TSI,-0.154492,0.177429,0.245528,0.199757,0.272046,0.255303,1.000000,0.052117
Aerosols,0.340238,-0.356155,-0.267809,-0.337055,-0.043921,-0.225131,0.052117,1.000000


## 2. Fitting A Model [1 point]

Fit a LASSO model with $\lambda = 0.06$ to the training data, including all variables except the year and month variables.  Print the coefficients for this model.

In [257]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.06, random_state=202411)
lasso.fit(X_train_scaled, y_train)
lasso_coefficients = lasso.coef_
lasso_coefficients

array([ 0.        ,  0.07989296,  0.        ,  0.00275763,  0.        ,
        0.        ,  0.        , -0.        ])

In [258]:
non_zero_coeffs = X_train.columns[lasso_coefficients != 0]
non_zero_coeffs

Index(['CO2', 'N2O'], dtype='object')

## 3. Picking a Best Model [2.5 points]

Instead of using a LASSO model, we decide that we'd rather move forward with a **ridge regression** model.  We don't know which $\lambda$ to use for this model, so let's explore which value of $\lambda$ might be most appropriate for a ridge regression model.

To do this, we'll explore $\lambda$ values between 0.05 and 1 exploring by every 0.05.  We can do this with code using:

`for m in range(1, 21):`

`alph = m / 20`

The following code sets up the folds for cross-validation.

In [259]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge

cross_val = KFold(n_splits=10, shuffle=True, random_state=202411)
r2_scores = []

**a)** Use 10-fold cross-validation to explore the $R^2$ values for each of the $\lambda$s as defined above.

In [260]:
for m in range(1, 21):
    alpha = m / 20
    ridge = Ridge(alpha=alpha)
    scores = cross_val_score(ridge, X_train_scaled, y_train, cv=cross_val, scoring='r2')
    r2_scores.append((alpha, scores.mean()))
    
r2_scores

[(0.05, 0.720615718439252),
 (0.1, 0.7206965131267431),
 (0.15, 0.7207455799211139),
 (0.2, 0.720768544253105),
 (0.25, 0.7207700181737827),
 (0.3, 0.7207537992105137),
 (0.35, 0.7207230264265524),
 (0.4, 0.7206803037292107),
 (0.45, 0.7206277979206412),
 (0.5, 0.7205673171297592),
 (0.55, 0.7205003739016763),
 (0.6, 0.7204282362123172),
 (0.65, 0.7203519689228166),
 (0.7, 0.7202724676217062),
 (0.75, 0.7201904863735195),
 (0.8, 0.7201066605647669),
 (0.85, 0.7200215257865711),
 (0.9, 0.7199355334987343),
 (0.95, 0.7198490640687936),
 (1.0, 0.7197624376614027)]

**b)** Print the $R^2$ values for each of the individual folds of your optimal $\lambda$.

In [261]:
optimal_lambda = max(r2_scores, key=lambda x: x[1])[0]
ridge = Ridge(alpha=optimal_lambda)
ridge_folds_r2 = cross_val_score(ridge, X_train_scaled, y_train, cv=cross_val, scoring='r2')
ridge_folds_r2

array([0.79338994, 0.81826251, 0.7369177 , 0.67382575, 0.73194208,
       0.7494757 , 0.43869564, 0.8173675 , 0.80106446, 0.64675889])

**c)** Repeat this process for a different set of 10-folds, using a random state of your choosing.  Determine which $\lambda$ results in the optimal mean $R^2$, similar to what you did above.

In [262]:
cross_val = KFold(n_splits=10, shuffle=True, random_state=42)
r2_scores = []

for i in range(1, 21):
    alpha = i / 20
    ridge = Ridge(alpha=alpha)
    scores = cross_val_score(ridge, X_train_scaled, y_train, cv=cross_val, scoring='r2')
    r2_scores.append((alpha, scores.mean()))

optimal_lambda = max(r2_scores, key=lambda x: x[1])[0]
optimal_lambda

0.15

In [263]:
ridge_folds_r2.min()

0.4386956427671851

In [264]:
ridge_folds_r2.max()

0.8182625066699701

In [265]:
cross_val_new = KFold(n_splits=10, shuffle=True, random_state=2024)
r2_scores_new = []

for i in range(1, 21):
    alpha = i / 20
    ridge = Ridge(alpha=alpha)
    scores = cross_val_score(ridge, X_train_scaled, y_train, cv=cross_val_new, scoring='r2')
    r2_scores_new.append((alpha, scores.mean()))

optimal_lambda_new = max(r2_scores_new, key=lambda x: x[1])[0]
optimal_lambda_new

0.65

## 4. Evaluating Our Best Model [1 point]

Refit the model with the optimal $\lambda$ found in Question **3b** to the full training data.  Print the resulting coefficients.

In [266]:
ridge_final = Ridge(alpha=optimal_lambda)
ridge_final.fit(X_train_scaled, y_train)
ridge_final_coefficients = ridge_final.coef_
ridge_final_coefficients

array([ 0.05949593,  0.07318252,  0.00637567, -0.06763033, -0.12815955,
        0.20541397,  0.03696694, -0.04615085])

In [267]:
ridge_final_coefficients[list(X_train.columns).index('N2O')]

-0.06763033274339218

In [268]:
ridge_final.score(X_test_scaled, y_test)

0.16181127386709993

Remember to keep all your cells and hit the save icon above periodically to checkpoint (save) your results on your local computer. Once you are satisified with your results restart the kernel and run all (Kernel -> Restart & Run All). **Make sure nothing has changed**. Checkpoint and exit (File -> Save and Checkpoint + File -> Close and Halt). Follow the instructions on the Homework 11 Canvas Assignment to submit your notebook to GitHub.